
# Advanced Graph Analysis & NLP

In this notebook, we will be using a combination of Natural Language Processing and network analysis to look at a Mafia network. The settings the mafia network is the following: As members of an investigation unit, we have been observing a network of families, which have been associated with nefarious activities. We will use two datasets:

1. PDF files written by an undercover agent
2. A network of interactions between those families

For NLP in Spark we can use the open-source *spark-nlp* library, which allows us to use a variety of Deep Learning models.

Since here we are given several PDFs to work with, we need a Python library to parse them using ``UDF``. We first need to install the external Python library ``pypdf``. The *graphframes* library, which we used in the previous lab, offers the useful ``GraphFrame``, but choices for graph algorithms are relatively limited. Thus, we will install the ``networkx`` library, which offers a range of popular graph algorithms.

In [ ]:
# #Checking the installed Java version
# !java -version
# !pip install "pyspark==3.5.0" 
# # Install Java 17
# !sudo apt-get update
# !sudo apt-get install -y openjdk-17-jdk-headless

# !java -version

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-124.04, mixed mode, sharing)
Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Get:2 https://cli.github.com/packages stable InRelease [3917 B]                
Hit:3 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://cloud.archive.ubuntu.com/ubuntu noble InRelease                  
Hit:8 http://deb.wakemeops.com/wakemeops stable InRelease                      
Hit:9 https://archive.ubuntu.com/ubuntu noble InRelease                        
Hit:10 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease         
Hit:11 https://cloud.archive.ubuntu.com

In [2]:
%pip install graphframes-py==0.10.0

Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip3 install pypdf networkx

In [4]:
import io
import numpy as np
import networkx as nx
from pypdf import PdfReader 

We install the ``spark-nlp`` dependencies next. 

In [5]:
!pip install spark-nlp==5.3.3

> This may take some time to run

In [6]:
import sparknlp

# Start Spark Session
spark = sparknlp.start()

from sparknlp.base import DocumentAssembler, Pipeline, LightPipeline
from sparknlp.annotator import (
    Tokenizer,
    WordEmbeddingsModel,
    NerDLModel,
    NerConverter
)

import pyspark.sql.functions as F

:: loading settings :: url = jar:file:/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/zeus/.ivy2/cache
The jars for the packages stored in: /home/zeus/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-87414ce1-c340-4df5-ab53-14e183fccc54;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;5.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	f

We load all the reports from our agent as PDF files.

In [7]:
mafia_network_communications = spark.read.format("binaryFile").load("/teamspace/studios/this_studio/week12/crime_letters/*.pdf")

In [8]:
mafia_network_communications.show(3)

+--------------------+--------------------+------+--------------------+
|                path|    modificationTime|length|             content|
+--------------------+--------------------+------+--------------------+
|file:/teamspace/s...|2025-11-24 16:55:...|142828|[25 50 44 46 2D 3...|
|file:/teamspace/s...|2025-11-24 16:55:...| 91833|[25 50 44 46 2D 3...|
|file:/teamspace/s...|2025-11-24 16:55:...| 89489|[25 50 44 46 2D 3...|
+--------------------+--------------------+------+--------------------+
only showing top 3 rows



As the next step, we define a Python UDF that takes the binary content of each PDF and convert it to text.

In [9]:

@F.udf
def pdf_to_text(pdf) -> str:
    """
    We transform a PDF (binary) into a string. The "content" column is already binary, so we read the bytes directly.
    """

    # First we load the binary content
    bytes_stream = io.BytesIO(pdf)

    # We initialize the reader
    reader = PdfReader(bytes_stream)

    # As the final step, we go over each page (note though that in our case our PDFs have only one page each) and extract the text
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"

    return text

In [10]:
mafia_network_communications = mafia_network_communications \
  .withColumn("report_text", pdf_to_text(F.col("content")))

In [11]:
mafia_network_communications.show()

+--------------------+--------------------+------+--------------------+--------------------+
|                path|    modificationTime|length|             content|         report_text|
+--------------------+--------------------+------+--------------------+--------------------+
|file:/teamspace/s...|2025-11-24 16:55:...|142828|[25 50 44 46 2D 3...|I am writing to r...|
|file:/teamspace/s...|2025-11-24 16:55:...| 91833|[25 50 44 46 2D 3...|I am writing to u...|
|file:/teamspace/s...|2025-11-24 16:55:...| 89489|[25 50 44 46 2D 3...|I am writing to p...|
|file:/teamspace/s...|2025-11-24 16:55:...| 27676|[25 50 44 46 2D 3...|I am writing to p...|
|file:/teamspace/s...|2025-11-24 16:55:...| 27587|[25 50 44 46 2D 3...|I am writing to r...|
|file:/teamspace/s...|2025-11-24 16:55:...| 27535|[25 50 44 46 2D 3...|I am writing to p...|
|file:/teamspace/s...|2025-11-24 16:55:...| 27517|[25 50 44 46 2D 3...|I am writing to p...|
+--------------------+--------------------+------+--------------------

#### Named Entity Recognition
Named Entity Recognition (NER) is a natural language processing (NLP) technique that identifies and categorizes named entities within text into predefined categories such as names of persons, organizations, locations and dates. Our informant, who has infiltrated the organization, is sending regular letters. You are tasked with building a prototype of automatically processing the reports and extracting the names of the people in each letter, which can then be related to the larger network. 

We will define manually a pipeline that transforms our report text first into an embedding representation and then extracts our entities.

> This may take some time to run

In [12]:
# Step 1: Transforms raw texts to "document" annotation
documentAssembler = DocumentAssembler()\
    .setInputCol("report_text")\
    .setOutputCol("document")

# Step 2: Tokenization
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

# Step 3: Get the embeddings using glove_100d
embeddings = WordEmbeddingsModel.pretrained("glove_100d").\
                  setInputCols(["document", "token"]).\
                  setOutputCol("embeddings")

# Step 4: Use the ``ner_dl`` model
public_ner = NerDLModel.pretrained("ner_dl", "en") \
          .setInputCols(["document", "token", "embeddings"]) \
          .setOutputCol("ner")

# Step 5: Convert to NER
ner_converter = NerConverter() \
                .setInputCols(["document", "token", "ner"]) \
                  .setOutputCol("entities")

# Define the pipeline
ner_pipeline = Pipeline(stages=[ documentAssembler, 
                                 tokenizer,
                                 embeddings,
                                 public_ner,
                                 ner_converter
                                 ])

glove_100d download started this may take some time.


Approximate size to download 145.3 MB
[ | ]glove_100d download started this may take some time.
Approximate size to download 145.3 MB
Download done! Loading the resource.
[OK!]
ner_dl download started this may take some time.
Approximate size to download 13.6 MB
[ | ]ner_dl download started this may take some time.
Approximate size to download 13.6 MB
Download done! Loading the resource.
[ / ]

2025-11-25 17:15:33.198234: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-25 17:15:33.293014: W external/org_tensorflow/tensorflow/core/common_runtime/colocation_graph.cc:1218] Failed to place the graph without changing the devices of some resources. Some of the operations (that had to be colocated with resource generating operations) are not supported on the resources' devices. Current candidate devices are [
  /job:localhost/replica:0/task:0/device:CPU:0].
See below for details of this colocation group:
Colocation Debug Info:
Colocation group had the following types and supported devices: 
Root Member(assigned_device_name_index_=-1 requested_device_name_='/device:GPU:0' assigned_device_nam

[OK!]


We once again make use of spark ``Pipelines``.

In [13]:
# We fit our model 
ner_pipeline_model = ner_pipeline.fit(mafia_network_communications)

# And transform the data
processed = ner_pipeline_model.transform(mafia_network_communications)

In [14]:
ner_results = processed \
    .select(F.col("ner"), F.col("path"))

Let us now inspect the results.

In [15]:
ner_results.show(1)

+--------------------+--------------------+
|                 ner|                path|
+--------------------+--------------------+
|[{named_entity, 0...|file:/teamspace/s...|
+--------------------+--------------------+
only showing top 1 row



To facilitate our analysis, we will use the ``path`` column to extract the date of the report and use the date to assign a report id. We will accomplish this by using a Window function. From there, we explode the array of struct (the result of the NER) and retrieve only the entity and the associated word.

In [16]:
from pyspark.sql.window import Window

In [17]:
# 1. We extract the date
# 2. Get a row ID based on date
# 3. Explode the column
# 4. Extract the results (in a struct)

reports_parsed = ner_results \
    .withColumn("date", F.to_date(F.regexp_extract(F.col("path"), r"(\d{4}-\d+-\d{2})", 1))) \
    .withColumn("report_id", F.row_number().over(Window.orderBy("date"))) \
    .withColumn("ner_exploded", F.explode("ner")) \
    .withColumns({
        "result":  F.col("ner_exploded.result"), 
        "metadata": F.col("ner_exploded.metadata.word") 
    }
    ) \
    .withColumn("row_number", F.row_number().over(Window.orderBy("report_id"))) \
    .select(F.col("result"), F.col("metadata"), F.col("report_id"), F.col("row_number"))

In [18]:
reports_parsed.show(50)

25/11/25 17:15:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 1

+------+--------------+---------+----------+
|result|      metadata|report_id|row_number|
+------+--------------+---------+----------+
|     O|             I|        1|         1|
|     O|            am|        1|         2|
|     O|       writing|        1|         3|
|     O|            to|        1|         4|
|     O|        report|        1|         5|
|     O|            on|        1|         6|
|     O|            my|        1|         7|
|     O|        recent|        1|         8|
|     O|  surveillance|        1|         9|
|     O|      findings|        1|        10|
|     O|     regarding|        1|        11|
|     O|       illegal|        1|        12|
|     O|    activities|        1|        13|
|     O|        within|        1|        14|
|     O|       various|        1|        15|
|     O|         crime|        1|        16|
|     O|      families|        1|        17|
|     O|             .|        1|        18|
| B-PER|        Giulia|        1|        19|
| I-PER|  

We know that our agent always spells out the full name of the persons he follows (how convenient!). This allows us to do a clever join to get the people's full name: We join the DataFrame onto itself on the newly created variable ``row_number``, where the left side corresponds to the first name and the right side to the last name. Since we know the ordering we have as the join key (``row_number``, ``row_number - 1``).

In [19]:
sub_network = reports_parsed.alias("df1").withColumnRenamed("metadata", "First Name").join(
    reports_parsed.alias("df2").withColumnRenamed("metadata", "Last Name"),
    (F.col("df1.row_number") == F.col("df2.row_number") - 1) & 
    (F.col("df1.result") == "B-PER") & 
    (F.col("df2.result") == "I-PER"),
    "inner"
)

sub_network.show()

25/11/25 17:15:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 1

+------+----------+---------+----------+------+---------+---------+----------+
|result|First Name|report_id|row_number|result|Last Name|report_id|row_number|
+------+----------+---------+----------+------+---------+---------+----------+
| B-PER|    Giulia|        1|        19| I-PER|  Bianchi|        1|        20|
| B-PER|  Federico|        1|        37| I-PER|   Romano|        1|        38|
| B-PER|     Elena|        1|        65| I-PER|    Conti|        1|        66|
| B-PER| Francesco|        1|        73| I-PER|    Ricci|        1|        74|
| B-PER|      Luca|        1|       113| I-PER|  Moretti|        1|       114|
| B-PER|  Giuseppe|        1|       139| I-PER|    Rossi|        1|       140|
| B-PER|     Carlo|        2|       280| I-PER|   Romano|        2|       281|
| B-PER|  Giovanni|        2|       308| I-PER|  Moretti|        2|       309|
| B-PER| Francesca|        2|       315| I-PER|   Marini|        2|       316|
| B-PER|  Giuseppe|        3|       394| I-PER|    R

We have now extracted a subnet of the overall Mafia network. Our tasks are now two-fold:

1. Which is the individual with the highest influence within the sub-network (based on each relation type)?
2. Which is the individual with the highest influence within the overall network (based on each relation type)?

Both of these question can be answered using network analysis!

In [20]:
from graphframes import * 

nodes = spark.read.csv("/teamspace/studios/this_studio/week12/mafia_nodes.csv", header=True)
edges = spark.read.csv("/teamspace/studios/this_studio/week12/mafia_edges.csv", header=True)


We will create an index column for the relation types.

In [21]:
import pyspark.sql.types as tp

edges = edges \
    .withColumn("relation_type_index", F.dense_rank().over(Window.orderBy("relation_type"))) \
    .withColumn("weight", F.col("weight").cast(tp.IntegerType()))
edges.show()

+---+---+--------------------+------+-------------------+
|src|dst|       relation_type|weight|relation_type_index|
+---+---+--------------------+------+-------------------+
| 11| 12|Asked for Meeting...|     2|                  1|
| 14| 13|Asked for Meeting...|     1|                  1|
| 20| 19|Asked for Meeting...|     1|                  1|
| 15| 13|Asked for Meeting...|     1|                  1|
|  4|  1|Asked for Meeting...|     1|                  1|
| 15| 14|Asked for Meeting...|     1|                  1|
|  5|  7|Asked for Meeting...|     1|                  1|
| 16| 13|Asked for Meeting...|     1|                  1|
|  7|  5|Asked for Meeting...|     1|                  1|
| 17| 18|Asked for Meeting...|     1|                  1|
|  9| 12|Asked for Meeting...|     1|                  1|
| 17| 19|Asked for Meeting...|     1|                  1|
| 12|  9|Asked for Meeting...|     1|                  1|
| 18| 17|Asked for Meeting...|     2|                  1|
| 13| 16|Asked

25/11/25 17:15:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Now we instantiate our complete graph.

In [22]:
mafia_graph = GraphFrame(nodes, edges)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


In [23]:
# Let us inspect the graph
mafia_graph.vertices.show()
mafia_graph.edges.show()

+---+----------+---------+--------------+
| id|First Name|Last Name|        Family|
+---+----------+---------+--------------+
|  1|  Giuseppe|    Rossi|  Rossi Family|
|  2|     Maria|    Rossi|  Rossi Family|
|  3|   Antonio|    Rossi|  Rossi Family|
|  4|     Sofia|    Rossi|  Rossi Family|
|  5|      Luca|  Bianchi|Bianchi Family|
|  6|    Giulia|  Bianchi|Bianchi Family|
|  7|     Marco|  Bianchi|Bianchi Family|
|  8|   Alessia|  Bianchi|Bianchi Family|
|  9| Francesco|    Ricci|  Ricci Family|
| 10|    Chiara|    Ricci|  Ricci Family|
| 11|   Roberto|    Ricci|  Ricci Family|
| 12|     Laura|    Ricci|  Ricci Family|
| 13|  Giovanni|  Moretti|Moretti Family|
| 14|      Anna|  Moretti|Moretti Family|
| 15|    Matteo|  Moretti|Moretti Family|
| 16|     Elena|  Moretti|Moretti Family|
| 17|     Carlo|   Romano| Romano Family|
| 18|     Lucia|   Romano| Romano Family|
| 19|  Federico|   Romano| Romano Family|
| 20|   Martina|   Romano| Romano Family|
+---+----------+---------+--------

25/11/25 17:15:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [24]:
import pyspark.sql.functions as F
mafia_graph.edges.select(F.col("relation_type")).distinct().show()

+--------------------+
|       relation_type|
+--------------------+
|Asked for Meeting...|
|          Threatened|
|          Sent Money|
|              Called|
+--------------------+



We now subset our overall ``nodes`` DataFrame to extract the sub-network only.

In [25]:
sub_network_nodes = mafia_graph.vertices \
    .join(sub_network, on=["First Name", "Last Name"], how="inner") \
    .dropDuplicates(["First Name", "Last Name"])

sub_network_nodes.show()

25/11/25 17:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 1

+----------+---------+---+--------------+------+---------+----------+------+---------+----------+
|First Name|Last Name| id|        Family|result|report_id|row_number|result|report_id|row_number|
+----------+---------+---+--------------+------+---------+----------+------+---------+----------+
|   Antonio|    Rossi|  3|  Rossi Family| B-PER|        7|       898| I-PER|        7|       899|
|     Carlo|   Romano| 17| Romano Family| B-PER|        2|       280| I-PER|        2|       281|
|    Chiara|    Ricci| 10|  Ricci Family| B-PER|        7|       906| I-PER|        7|       907|
|     Elena|    Conti| 26|  Conti Family| B-PER|        1|        65| I-PER|        1|        66|
|  Federico|   Romano| 19| Romano Family| B-PER|        1|        37| I-PER|        1|        38|
| Francesca|   Marini| 24| Marini Family| B-PER|        2|       315| I-PER|        2|       316|
| Francesco|    Ricci|  9|  Ricci Family| B-PER|        1|        73| I-PER|        1|        74|
|  Giovanni|  Morett

In [26]:
# Get unique edges from subnetwork
sub_network_edges = list(map(lambda x: x["id"], sub_network_nodes.select("id").collect()))

# Filter network by subnetwork nodes
sub_network_df = mafia_graph.filterEdges(F.col("src").isin(sub_network_edges) | F.col("dst").isin(sub_network_edges)).edges

25/11/25 17:15:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 1

In [27]:
mafia_subgraph = GraphFrame(sub_network_nodes, sub_network_df)

mafia_subgraph.vertices.show()

25/11/25 17:15:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 1

+----------+---------+---+--------------+------+---------+----------+------+---------+----------+
|First Name|Last Name| id|        Family|result|report_id|row_number|result|report_id|row_number|
+----------+---------+---+--------------+------+---------+----------+------+---------+----------+
|   Antonio|    Rossi|  3|  Rossi Family| B-PER|        7|       898| I-PER|        7|       899|
|     Carlo|   Romano| 17| Romano Family| B-PER|        2|       280| I-PER|        2|       281|
|    Chiara|    Ricci| 10|  Ricci Family| B-PER|        7|       906| I-PER|        7|       907|
|     Elena|    Conti| 26|  Conti Family| B-PER|        1|        65| I-PER|        1|        66|
|  Federico|   Romano| 19| Romano Family| B-PER|        1|        37| I-PER|        1|        38|
| Francesca|   Marini| 24| Marini Family| B-PER|        2|       315| I-PER|        2|       316|
| Francesco|    Ricci|  9|  Ricci Family| B-PER|        1|        73| I-PER|        1|        74|
|  Giovanni|  Morett

While ``graphframes`` offers a method to compute ``degree centrality``, its inventory is relatively limited. Since we are operating on different partitions of the graph (i.e. the subgraph induced by the mentioned people in the reports and the entire graph), we can use Spark's capabilities and parallelize the operations. To this end, we will make use of the ``networkx`` library, which offers a wealth of functions to work with graphs.

As we can see, there are four different types of edges:
- Asked for Meeting
- Threatened
- Sent Money
- Called

Those edge types give us the idea that this graph is directed.

Let us now proceed to our actual network analysis. We will compute to network centrality measures here, (in-/out-)degree centrality and betweenness centrality.

- *Degree centrality* measures the importance of a node in a network based on its connections. In the context of in-degree centrality, this metric quantifies how many incoming connections a node has, reflecting its popularity or influence within the network. Conversely, out-degree centrality assesses the number of outgoing connections from a node, indicating its capacity to disseminate information or influence others. 

- *Betweenness centrality*, on the other hand, evaluates the extent to which a node serves as a bridge or intermediary between other nodes in the network. Nodes with high betweenness centrality often lie on many shortest paths between pairs of nodes, suggesting their critical role in maintaining connectivity and facilitating communication within the network.

Both of these, among many others, are implement in ``networkx``. To make us of these functionalities, we use a trick we learned in a previous class: ``pandas`` UDFs, which allow us to pass our graphframe or dataframe into a function and operate on it as normal Python code.

In [28]:
import pandas as pd

output_schema_degree = tp.StructType([
    tp.StructField("relation_type", tp.StringType(), False),
    tp.StructField("node", tp.StringType(), False),
    tp.StructField("in_degree_centrality", tp.FloatType(), False),
    tp.StructField("out_degree_centrality", tp.FloatType(), False),
])

output_schema_betweenness = tp.StructType([
    tp.StructField("relation_type", tp.StringType(), False),
    tp.StructField("node", tp.StringType(), False),
    tp.StructField("betweenness_centrality", tp.FloatType(), False),
])

def nx_degree_centrality(pdf: pd.DataFrame) -> pd.DataFrame:
    # We get the relation_type key
    key = pdf["relation_type"].iloc[0]

    # Here we instantiate a networkx directed graph (DiGraph)
    in_degree_centralities = nx.in_degree_centrality(nx.DiGraph(nx.from_pandas_edgelist(pdf, "src", "dst", edge_attr="weight")))
    out_degree_centralities = nx.out_degree_centrality(nx.DiGraph(nx.from_pandas_edgelist(pdf, "src", "dst", edge_attr="weight")))
    
    # Finally, we return a pd.DataFrame
    return pd.DataFrame(
        {
            "relation_type": [key for _ in range(1, len(in_degree_centralities.values()) + 1)], 
            "node": in_degree_centralities.keys(), 
            "in_degree_centrality": in_degree_centralities.values(),
            "out_degree_centrality": out_degree_centralities.values(),
        }
    )

def nx_betweenness_centrality(pdf: pd.DataFrame) -> pd.DataFrame:
    # We get the relation_type key
    key = pdf["relation_type"].iloc[0]

    # Here we instantiate a networkx directed graph (DiGraph)
    betweenness_centrality = nx.betweenness_centrality(nx.DiGraph(nx.from_pandas_edgelist(pdf, "src", "dst", edge_attr="weight")))
    
    # Finally, we return a pd.DataFrame
    return pd.DataFrame(
        {
            "relation_type": [key for _ in range(1, len(betweenness_centrality.values()) + 1)], 
            "node": betweenness_centrality.keys(), 
            "betweenness_centrality": betweenness_centrality.values()
        }
    )

We compute degree centrality and betweenness centrality for the subgraph

In [29]:
# Summarize results for degree centrality
mafia_subgraph \
    .edges \
    .groupby("relation_type") \
    .applyInPandas(nx_degree_centrality, output_schema_degree) \
    .alias("degree_df") \
    .join(mafia_graph.vertices.alias("node_df"), F.col("degree_df.node") == F.col("node_df.id")) \
    .orderBy("relation_type", "in_degree_centrality", ascending=False) \
    .show()

25/11/25 17:15:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+----+--------------------+---------------------+---+----------+---------+--------------+
|relation_type|node|in_degree_centrality|out_degree_centrality| id|First Name|Last Name|        Family|
+-------------+----+--------------------+---------------------+---+----------+---------+--------------+
|   Threatened|   2|          0.14285715|           0.14285715|  2|     Maria|    Rossi|  Rossi Family|
|   Threatened|   1|          0.14285715|           0.14285715|  1|  Giuseppe|    Rossi|  Rossi Family|
|   Threatened|   4|          0.14285715|           0.14285715|  4|     Sofia|    Rossi|  Rossi Family|
|   Threatened|   3|          0.14285715|           0.14285715|  3|   Antonio|    Rossi|  Rossi Family|
|   Threatened|  10|          0.14285715|           0.14285715| 10|    Chiara|    Ricci|  Ricci Family|
|   Threatened|   9|          0.14285715|           0.14285715|  9| Francesco|    Ricci|  Ricci Family|
|   Threatened|  13|          0.14285715|           0.14285715| 

In [30]:
# Summarize results for betweenness centrality
mafia_subgraph \
    .edges \
    .groupby("relation_type") \
    .applyInPandas(nx_betweenness_centrality, output_schema_betweenness) \
    .alias("degree_df") \
    .join(mafia_graph.vertices.alias("node_df"), F.col("degree_df.node") == F.col("node_df.id")) \
    .orderBy("relation_type", "betweenness_centrality", ascending=False) \
    .show()

25/11/25 17:15:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+----+----------------------+---+----------+---------+--------------+
|relation_type|node|betweenness_centrality| id|First Name|Last Name|        Family|
+-------------+----+----------------------+---+----------+---------+--------------+
|   Threatened|   2|                   0.0|  2|     Maria|    Rossi|  Rossi Family|
|   Threatened|   1|                   0.0|  1|  Giuseppe|    Rossi|  Rossi Family|
|   Threatened|   4|                   0.0|  4|     Sofia|    Rossi|  Rossi Family|
|   Threatened|   3|                   0.0|  3|   Antonio|    Rossi|  Rossi Family|
|   Threatened|  10|                   0.0| 10|    Chiara|    Ricci|  Ricci Family|
|   Threatened|   9|                   0.0|  9| Francesco|    Ricci|  Ricci Family|
|   Threatened|  13|                   0.0| 13|  Giovanni|  Moretti|Moretti Family|
|   Threatened|  17|                   0.0| 17|     Carlo|   Romano| Romano Family|
|   Sent Money|   1|             0.6929429|  1|  Giuseppe|    Rossi|  Rossi 

We repeat this exercise for the entire graph.

In [31]:
# Summarize results for degree centrality
mafia_graph \
    .edges \
    .groupby("relation_type") \
    .applyInPandas(nx_degree_centrality, output_schema_degree) \
    .alias("degree_df") \
    .join(mafia_graph.vertices.alias("node_df"), F.col("degree_df.node") == F.col("node_df.id")) \
    .orderBy("relation_type", "in_degree_centrality", ascending=False) \
    .show()

25/11/25 17:15:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+----+--------------------+---------------------+---+----------+---------+--------------+
|relation_type|node|in_degree_centrality|out_degree_centrality| id|First Name|Last Name|        Family|
+-------------+----+--------------------+---------------------+---+----------+---------+--------------+
|   Threatened|   2|          0.07692308|           0.07692308|  2|     Maria|    Rossi|  Rossi Family|
|   Threatened|   1|          0.07692308|           0.07692308|  1|  Giuseppe|    Rossi|  Rossi Family|
|   Threatened|   4|          0.07692308|           0.07692308|  4|     Sofia|    Rossi|  Rossi Family|
|   Threatened|   3|          0.07692308|           0.07692308|  3|   Antonio|    Rossi|  Rossi Family|
|   Threatened|  10|          0.07692308|           0.07692308| 10|    Chiara|    Ricci|  Ricci Family|
|   Threatened|   9|          0.07692308|           0.07692308|  9| Francesco|    Ricci|  Ricci Family|
|   Threatened|  12|          0.07692308|           0.07692308| 

In [32]:
# Summarize results for betweenness centrality
mafia_graph \
    .edges \
    .groupby("relation_type") \
    .applyInPandas(nx_betweenness_centrality, output_schema_betweenness) \
    .alias("degree_df") \
    .join(mafia_graph.vertices.alias("node_df"), F.col("degree_df.node") == F.col("node_df.id")) \
    .orderBy("relation_type", "betweenness_centrality", ascending=False) \
    .show()

25/11/25 17:15:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/25 17:15:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+----+----------------------+---+----------+---------+--------------+
|relation_type|node|betweenness_centrality| id|First Name|Last Name|        Family|
+-------------+----+----------------------+---+----------+---------+--------------+
|   Threatened|   2|                   0.0|  2|     Maria|    Rossi|  Rossi Family|
|   Threatened|   1|                   0.0|  1|  Giuseppe|    Rossi|  Rossi Family|
|   Threatened|   4|                   0.0|  4|     Sofia|    Rossi|  Rossi Family|
|   Threatened|   3|                   0.0|  3|   Antonio|    Rossi|  Rossi Family|
|   Threatened|  10|                   0.0| 10|    Chiara|    Ricci|  Ricci Family|
|   Threatened|   9|                   0.0|  9| Francesco|    Ricci|  Ricci Family|
|   Threatened|  12|                   0.0| 12|     Laura|    Ricci|  Ricci Family|
|   Threatened|  11|                   0.0| 11|   Roberto|    Ricci|  Ricci Family|
|   Threatened|  30|                   0.0| 30|     Paola|    Rossi|  Rossi 

#### Bonus
If would like to compute the overall influence, without regard to the relation type, you can use a *fictional* relation type by creating a column with a literal value, such as 1 or a string. Note that using networkx does not leverage the speed of Spark, unless we partition our network in some way. Since we have a network with weights, we can use a groupby to sum the weight, which automatically imposes a uniqueness condition. We still need our fictional groupby to use ``applyInPandas``.

In [33]:
mafia_graph \
    .edges \
    .withColumn("relation_type", F.lit("1")) \
    .groupby(["src", "dst", "relation_type"]) \
    .agg(F.sum("weight").alias("weight")) \
    .groupby("relation_type") \
    .applyInPandas(nx_degree_centrality, output_schema_degree) \
    .alias("degree_df") \
    .join(mafia_graph.vertices.alias("node_df"), F.col("degree_df.node") == F.col("node_df.id")) \
    .orderBy("relation_type", "in_degree_centrality", ascending=False) \
    .show()

+-------------+----+--------------------+---------------------+---+----------+---------+--------------+
|relation_type|node|in_degree_centrality|out_degree_centrality| id|First Name|Last Name|        Family|
+-------------+----+--------------------+---------------------+---+----------+---------+--------------+
|            1|   9|          0.29090908|           0.29090908|  9| Francesco|    Ricci|  Ricci Family|
|            1|  13|          0.29090908|           0.29090908| 13|  Giovanni|  Moretti|Moretti Family|
|            1|   1|          0.29090908|           0.29090908|  1|  Giuseppe|    Rossi|  Rossi Family|
|            1|   3|          0.29090908|           0.29090908|  3|   Antonio|    Rossi|  Rossi Family|
|            1|  11|          0.27272728|           0.27272728| 11|   Roberto|    Ricci|  Ricci Family|
|            1|   6|          0.27272728|           0.27272728|  6|    Giulia|  Bianchi|Bianchi Family|
|            1|   8|          0.27272728|           0.27272728| 